### Mapping points from Torus to $R^3$

In [11]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [5]:
def map_torus_to_r3(point, R, r):
    """
    Maps a point from the interior of a torus to R^3.

    Args:
        point (tuple or list): The (x, y, z) coordinates of the point inside the torus.
        inner_radius_a (float): The inner radius of the torus.
        outer_radius_b (float): The outer radius of the torus.

    Returns:
        numpy.ndarray: The transformed (x', y', z') coordinates in R^3.
    """
    x, y, z = point

    # 1. Calculate the major and minor radii of the torus
    #R = (inner_radius_a + outer_radius_b) / 2.0
    #r = (outer_radius_b - inner_radius_a) / 2.0

    # Handle the case where the point is on the z-axis to avoid division by zero
    r_xy = np.sqrt(x**2 + y**2)
    if r_xy == 0:
        # If the point is on the z-axis, its projection on the xy-plane is the origin.
        # We can define the corresponding point C on the central circle to be (R, 0, 0).
        C = np.array([R, 0, 0])
    else:
        # 2. Find the corresponding point C on the central circle (in the xy-plane)
        C = np.array([R * x / r_xy, R * y / r_xy, 0])

    # 3. Determine the position vector V within the cross-section
    P = np.array([x, y, z])
    V = P - C

    # 4. Calculate the distance 's' of the point from the central circle
    s = np.linalg.norm(V)

    # If the point is on the central circle, it does not move.
    if s == 0:
        return P

    # 5. Define the scaling function to map the distance to infinity
    # This stretches the distance from the range [0, r) to [0, infinity)
    s_prime = np.tan((np.pi / 2) * (s / r))

    # 6. Construct the transformed point P'
    # The direction from C to P is given by the unit vector V/s
    P_prime = C + s_prime * (V / s)

    return P_prime

In [4]:
state_vectors = np.loadtxt("NEOS_state_vectors.txt")
state_vectors

array([[ 1.50564476e+00, -8.24086449e-01,  1.49155679e-01,
         1.48131740e+00,  4.01462703e+00,  6.66810679e-01],
       [-4.97321553e-01,  2.47466938e+00, -5.12774515e-01,
        -3.67123769e+00,  1.44378549e+00, -3.45684097e-01],
       [ 1.86254335e+00, -3.05239693e+00, -1.12659812e-01,
         1.39433630e+00,  2.01164521e+00, -3.32517926e-01],
       ...,
       [-8.27481539e-01,  6.47049584e-01,  1.55307411e-04,
        -1.66731661e+00, -5.86126182e+00,  1.81401780e+00],
       [-1.26385772e+00,  8.43129403e-01,  1.48471976e-01,
        -6.40888573e-01, -3.07389320e+00,  2.57901055e+00],
       [-3.19994846e+00, -3.05371847e+00,  2.36845340e-01,
         6.56921545e-01, -1.83756004e+00,  1.52884727e-01]])

In [8]:
velocity_vectors = state_vectors[:,3:]

#parametros del toro con 95% de los datos
R = 5.5044
r = 3.6063
transformed_points = np.array([map_torus_to_r3(v, R, r) for v in velocity_vectors])

In [17]:
s_distances = np.sqrt((np.sqrt(velocity_vectors[:,0]**2 + velocity_vectors[:,1]**2) - R)**2 + velocity_vectors[:,2]**2)
# 4. Create the Plotly Figure
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}]],
    subplot_titles=('Original Points in Torus', 'Transformed Points in R³')
)

# --- First Plot: Original Torus Points ---
fig.add_trace(
    go.Scatter3d(
        x=velocity_vectors[:, 0],
        y=velocity_vectors[:, 1],
        z=velocity_vectors[:, 2],
        mode='markers',
        marker=dict(
            size=2,
            color=s_distances,  # Color by distance from the central axis
            colorscale='Viridis',
            colorbar=dict(title='Distance from Center of Tube'),
            showscale=True
        )
    ),
    row=1, col=1
)

# --- Second Plot: Transformed R3 Points ---
fig.add_trace(
    go.Scatter3d(
        x=transformed_points[:, 0],
        y=transformed_points[:, 1],
        z=transformed_points[:, 2],
        mode='markers',
        marker=dict(
            size=2,
            color=s_distances,  # Use the same color scale
            colorscale='Viridis',
            showscale=False # Color bar is already shown
        )
    ),
    row=1, col=2
)

# --- Update Layout and Scene Properties ---
fig.update_layout(
    title_text='Transformation from Torus Interior to R³',
    height=700,
    # Make the torus plot have a correct aspect ratio
    scene1=dict(
        xaxis_title='X', yaxis_title='Y', zaxis_title='Z',
        aspectmode='data' # This ensures the torus is not distorted
    ),
    scene2=dict(
        xaxis_title='X', yaxis_title='Y', zaxis_title='Z', zaxis=dict(range=[-2000, 2000]), xaxis=dict(range=[-2000, 2000]), yaxis=dict(range=[-2000, 2000]),
        aspectmode='auto' # The R3 plot can auto-scale
    )
)

fig.show()

In [20]:
#parametros del toro con 97% de los datos

R = 5.5044
r = 5.8697
transformed_points = np.array([map_torus_to_r3(v, R, r) for v in velocity_vectors])

s_distances = np.sqrt((np.sqrt(velocity_vectors[:,0]**2 + velocity_vectors[:,1]**2) - R)**2 + velocity_vectors[:,2]**2)
# 4. Create the Plotly Figure
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}]],
    subplot_titles=('Original Points in Torus', 'Transformed Points in R³')
)

# --- First Plot: Original Torus Points ---
fig.add_trace(
    go.Scatter3d(
        x=velocity_vectors[:, 0],
        y=velocity_vectors[:, 1],
        z=velocity_vectors[:, 2],
        mode='markers',
        marker=dict(
            size=2,
            color=s_distances,  # Color by distance from the central axis
            colorscale='Viridis',
            colorbar=dict(title='Distance from Center of Tube'),
            showscale=True
        )
    ),
    row=1, col=1
)

# --- Second Plot: Transformed R3 Points ---
fig.add_trace(
    go.Scatter3d(
        x=transformed_points[:, 0],
        y=transformed_points[:, 1],
        z=transformed_points[:, 2],
        mode='markers',
        marker=dict(
            size=2,
            color=s_distances,  # Use the same color scale
            colorscale='Viridis',
            showscale=False # Color bar is already shown
        )
    ),
    row=1, col=2
)

# --- Update Layout and Scene Properties ---
fig.update_layout(
    title_text='Transformation from Torus Interior to R³',
    height=700,
    # Make the torus plot have a correct aspect ratio
    scene1=dict(
        xaxis_title='X', yaxis_title='Y', zaxis_title='Z',
        aspectmode='data' # This ensures the torus is not distorted
    ),
    scene2=dict(
        xaxis_title='X', yaxis_title='Y', zaxis_title='Z', zaxis=dict(range=[-2000, 2000]), xaxis=dict(range=[-2000, 2000]), yaxis=dict(range=[-2000, 2000]),
        aspectmode='auto' # The R3 plot can auto-scale
    )
)

fig.show()

### Jacobian Trasform from Torus To $R^3$

$$T(v_x,v_y,v_z) = (v_x',v_y',v_z')$$

$$J_T = [[∂x'/∂x, ∂x'/∂y, ∂x'/∂z], [∂y'/∂x, ∂y'/∂y, ∂y'/∂z], [∂z'/∂x, ∂z'/∂y, ∂z'/∂z]]$$

In [ ]:
def calculate_jacobian_torus_to_r3(point, R, r):
    """
    Calculates the Jacobian matrix of the transformation from Torus to R^3 at a given point.
    """
    x, y, z = point
    #R = (a + b) / 2.0
    #r = (b - a) / 2.0

    # Handle singularity at the z-axis
    r_xy = np.sqrt(x**2 + y**2)
    if r_xy < 1e-9: # Avoid division by zero
        # The Jacobian is not well-defined on the z-axis, but we can return a limit
        # For simplicity, we'll state it's undefined for this example.
        print("Warning: Jacobian is not well-defined on the z-axis.")
        return np.full((3, 3), np.nan)

    s = np.sqrt((r_xy - R)**2 + z**2)
    if s < 1e-9: # Avoid division by zero at the central circle
        print("Warning: Jacobian is not well-defined on the central circle.")
        return np.full((3, 3), np.nan)

    # Scaling function and its derivative
    f_s = np.tan((np.pi / 2.0) * (s / r))
    f_prime_s = (np.pi / (2.0 * r)) * (1.0 / np.cos((np.pi / 2.0) * (s / r))**2)

    # Pre-compute common terms
    term1 = (1 - R / r_xy)
    term2 = f_prime_s - f_s / s

    # Initialize Jacobian matrix
    J = np.zeros((3, 3))

    # Partial derivatives (dx'/dx, dx'/dy, dx'/dz)
    J[0, 0] = (R/r_xy) * (y**2/r_xy**2) + (f_s/s)*term1 + (x**2/s**2) * term1**2 * term2
    J[0, 1] = -(R/r_xy) * (x*y/r_xy**2) + (x*y/s**2) * term1**2 * term2
    J[0, 2] = (x*z/s**2) * ((r_xy - R)/r_xy) * term2

    # Partial derivatives (dy'/dx, dy'/dy, dy'/dz)
    J[1, 0] = -(R/r_xy) * (x*y/r_xy**2) + (x*y/s**2) * term1**2 * term2
    J[1, 1] = (R/r_xy) * (x**2/r_xy**2) + (f_s/s)*term1 + (y**2/s**2) * term1**2 * term2
    J[1, 2] = (y*z/s**2) * ((r_xy - R)/r_xy) * term2

    # Partial derivatives (dz'/dx, dz'/dy, dz'/dz)
    J[2, 0] = (x*z/s**2) * ((r_xy - R)/r_xy) * term2
    J[2, 1] = (y*z/s**2) * ((r_xy - R)/r_xy) * term2
    J[2, 2] = (f_s/s) + (z**2/s**2) * term2
    
    return J

In [23]:
for v in velocity_vectors:
    J = calculate_jacobian_torus_to_r3(v, R, r)
    det = np.linalg.det(J)
    inv_det = 1/det
    print("Jacobian determinant: ", det)
    print("Inverse Jacobian determinant: ", inv_det)

Jacobian determinant:  10.446393833921675
Inverse Jacobian determinant:  0.09572681404685185
Jacobian determinant:  -4.153277941231843
Inverse Jacobian determinant:  -0.24077367663561777
Jacobian determinant:  -1.9002860851747443
Inverse Jacobian determinant:  -0.5262365534335022
Jacobian determinant:  -1213538.8818579726
Inverse Jacobian determinant:  -8.240362257441338e-07
Jacobian determinant:  3.386274321038699
Inverse Jacobian determinant:  0.29530980221745945
Jacobian determinant:  -2.246611458273357
Inverse Jacobian determinant:  -0.4451147955813215
Jacobian determinant:  -51395.738630273794
Inverse Jacobian determinant:  -1.945686600972336e-05
Jacobian determinant:  11.806126451110465
Inverse Jacobian determinant:  0.0847017863260258
Jacobian determinant:  69.62524925456691
Inverse Jacobian determinant:  0.01436260567403868
Jacobian determinant:  -0.950546288576852
Inverse Jacobian determinant:  -1.0520266209204705
Jacobian determinant:  -0.9038786904421411
Inverse Jacobian det